# Preparación del modelado — tabla, particiones y métricas

## Qué responde este notebook

Antes de comparar modelos hay que fijar cuatro cosas, para que los notebooks siguientes no las
repitan ni las cambien a mitad de camino:

1. **La tabla y las variables**: las 25 que se eligieron en
   [`3.3_seleccion_estable.ipynb`](3.3_seleccion_estable.ipynb), en dos versiones según la familia
   de modelo.
2. **Las particiones en el tiempo**: entrenamiento, validación (donde se elige) y prueba (que solo se
   reporta).
3. **Cómo se mide y cómo se elige**: qué métrica decide en cada resultado y qué requisitos debe
   cumplir un modelo para ser elegible, con base en la literatura de evaluación de pronósticos.
4. **Los valores faltantes**: por qué existen y cómo los maneja cada familia de modelo.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling, experimentos

import pandas as pd

pd.set_option("display.width", 160)

wr = features.construir_tabla_modelado(range(2016, 2026))
print("filas:", wr.shape[0], "columnas:", wr.shape[1])

filas: 23610 columnas: 203


## 1. Las variables, en dos versiones

`features.columnas_por_modelo("arbol")` devuelve las 25 variables de
[`3.3_seleccion_estable.ipynb`](3.3_seleccion_estable.ipynb): el conjunto más chico que, con 95% de
confianza, no es más de 1% peor que el mejor en validación, en el orden estable de ese notebook.
`columnas_por_modelo("ridge")` es la versión para el modelo lineal: quita las variantes de
`target_share` y `air_yards_share`, muy correlacionadas con `wopr`, y agrega la experiencia en sus
versiones lineal, cuadrática y por tramos (`anios_experiencia`, `anios_experiencia_sq`,
`anios_experiencia_bucket`), que un modelo lineal necesita para captar la curva de la experiencia y
un árbol no. Qué mide cada variable
está en el [glosario](../../../docs/data/GLOSARIO_VARIABLES.md) (sección 8).

In [2]:
features_arbol = features.columnas_por_modelo("arbol")
features_ridge = features.columnas_por_modelo("ridge")
print(f"arbol: {len(features_arbol)} columnas")
print(f"ridge: {len(features_ridge)} columnas ({len(set(features_arbol) - set(features_ridge))} se quitan, "
      f"{len(set(features_ridge) - set(features_arbol))} se agregan)")
faltantes = [c for c in set(features_arbol) | set(features_ridge) if c not in wr.columns]
print("columnas declaradas pero ausentes en la tabla (debe ser vacío):", faltantes)

arbol: 25 columnas
ridge: 24 columnas (4 se quitan, 3 se agregan)
columnas declaradas pero ausentes en la tabla (debe ser vacío): []


## 2. Tres particiones en el tiempo

Las particiones son por temporada, nunca al azar: si se mezclaran semanas de todos los años, el
modelo aprendería con partidos posteriores a los que después tiene que predecir.

- **Entrenamiento**: 2016-2021 (6 temporadas).
- **Validación**: 2022-2023. Aquí se eligen el modelo y sus hiperparámetros.
- **Prueba**: 2024-2025. Se mira una sola vez, al final, para reportar el resultado.

Una salvedad: la selección de variables de `3.3_seleccion_estable.ipynb` se decidió sin la prueba,
pero su regla para elegir el tamaño del conjunto se corrigió después de reportar la prueba (ver
[`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)). Por eso la verificación sin ningún uso previo es
la temporada 2026, en `6.1_modelo_final_y_temporada_actual.ipynb`.

In [3]:
TRAIN_SEASONS = range(2016, 2022)
VAL_SEASONS = [2022, 2023]
TEST_SEASONS = [2024, 2025]

train_mask, val_mask, test_mask = experimentos.dividir_temporal(wr, TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS)
print(f"entrenamiento: {train_mask.sum()} filas (2016-2021)")
print(f"validacion:    {val_mask.sum()} filas (2022-2023)")
print(f"prueba:        {test_mask.sum()} filas (2024-2025)")
print(f"filas fuera de las 3 particiones: {(~(train_mask | val_mask | test_mask)).sum()}")

entrenamiento: 13795 filas (2016-2021)
validacion:    4863 filas (2022-2023)
prueba:        4952 filas (2024-2025)
filas fuera de las 3 particiones: 0


## 3. Cómo se mide y cómo se elige

Los modelos predicen un **valor esperado**: cuántas recepciones, yardas o touchdowns tendrá en
promedio un receptor con ese perfil. Es el valor que se suma para armar puntos de fantasy, así que la
métrica que decide tiene que premiar justo eso. Gneiting (2011) muestra que el error cuadrático, y
las métricas que salen de él (RMSE y R²), premia predecir la **media**, mientras que el error
absoluto (MAE) premia predecir la **mediana**. Elegir con una métrica que no corresponde a lo que se
predice lleva a conclusiones equivocadas.

Un ejemplo con los datos de validación, usando predicciones que no miran ninguna variable: el
promedio de entrenamiento, la mediana de entrenamiento y el cero.

In [4]:
media_train = {t: wr.loc[train_mask, t].mean() for t in modeling.TARGETS}
filas = {}
for t in modeling.TARGETS:
    y_val = wr.loc[val_mask, t]
    constantes = {
        "media de entrenamiento": media_train[t],
        "mediana de entrenamiento": wr.loc[train_mask, t].median(),
        "cero": 0.0,
    }
    for nombre, valor in constantes.items():
        m = modeling.metricas_target(t, y_val, [valor] * len(y_val), media_entrenamiento=media_train[t])
        filas[(t, f"predecir {nombre} ({valor:.2f})")] = {k: m.get(k) for k in ("mae", "rmse", "r2_oos", "deviance_poisson", "d2_oos")}
pd.DataFrame(filas).T.round(4)

mae     rmse  r2_oos  deviance_poisson   d2_oos
receptions      predecir media de entrenamiento (2.76)      2.0628   2.5729  0.0000            2.5411   0.0000
                predecir mediana de entrenamiento (2.00)    1.9560   2.6630 -0.0713            2.7542  -0.0839
                predecir cero (0.00)                        2.6909   3.7223 -1.0931           76.8379 -29.2383
receiving_yards predecir media de entrenamiento (35.35)    29.3350  37.0354  0.0000               NaN      NaN
                predecir mediana de entrenamiento (25.00)  27.7783  38.0577 -0.0560               NaN      NaN
                predecir cero (0.00)                       33.9060  50.1751 -0.8355               NaN      NaN
receiving_tds   predecir media de entrenamiento (0.22)      0.3355   0.4539  0.0000            0.7026   0.0000
                predecir mediana de entrenamiento (0.00)    0.1929   0.4926 -0.1779            5.0089  -6.1295
                predecir cero (0.00)                        0.1929   0.4926 -0.1779            5.0089  -6.1295

**Lectura.** En recepciones y yardas, la mediana gana en MAE (1.956 contra 2.063 en recepciones;
27.78 contra 29.34 en yardas) y la media gana en RMSE (2.573 contra 2.663; 37.04 contra 38.06). Es
justo la diferencia entre premiar la mediana y premiar la media. En touchdowns la mediana es 0, y
predecir 0 para todos tiene el mejor MAE de la tabla (0.193 contra 0.336 de la media), pero su D² es
−6.13: como valor esperado es la peor predicción posible. Si se eligiera por MAE, un modelo que
predice cerca de 0 ganaría sin aportar nada.

### La política de métricas

| Resultado | Métrica que decide (en validación) | También se reporta |
|---|---|---|
| `receptions` | RMSE | MAE, deviance de Poisson, R² fuera de muestra, sesgo, pendiente de calibración, Spearman semanal |
| `receiving_yards` | RMSE | MAE, R² fuera de muestra, sesgo, pendiente de calibración, Spearman semanal |
| `receiving_tds` | Deviance de Poisson | D² y R² fuera de muestra, Brier y AUC de «anota al menos uno», sesgo, pendiente, RMSE; MAE solo informativo |

- **RMSE** premia la media y se expresa en las unidades del resultado.
- **Deviance de Poisson** es la medida de error propia de los conteos (Czado, Gneiting y Held, 2009)
  y la principal en la práctica actuarial, donde los conteos también están llenos de ceros. Castiga
  más predecir casi cero cuando el evento sí ocurre.
- **R² y D² fuera de muestra** comparan contra la media de entrenamiento, lo único que se conocería
  antes de predecir (Campbell y Thompson, 2008). Un valor negativo significa que el modelo es peor que
  el promedio histórico. D² es lo mismo que R², con deviance en lugar de error cuadrático.
- **Sesgo y pendiente de calibración** dicen si las predicciones están en la escala correcta: sesgo
  cero y pendiente uno. Gneiting y Resin (2023) muestran que R² = (discriminación − descalibración) /
  incertidumbre, así que un modelo que ordena bien pero está descalibrado puede tener R² negativa. En
  decisiones deportivas la calibración pesa más que la exactitud (Walsh y Joshi, 2024).
- **Spearman semanal** mide si el modelo ordena bien a los receptores dentro de cada semana, que es
  como se evalúa a los expertos de fantasy (FantasyPros).
- **MAE** se reporta porque es fácil de leer, pero no decide: en un conteo con mayoría de ceros,
  predecir cero lo minimiza (Kolassa, 2016), como mostró la tabla de arriba.

**Empates.** Si la diferencia entre dos modelos en la métrica que decide no se distingue del
ruido, no hay ganador real. Se mide con la pregunta de Diebold y Mariano (1995), si dos pronósticos
tienen la misma capacidad predictiva, mediante un intervalo de 95% por bootstrap de semanas completas
(`experimentos.diferencia_bootstrap`). Si el intervalo incluye cero se elige XGBoost, la familia que
aplica el flujo semanal (`weekly/wr/src/pipeline.py`, que predice cada semana con los modelos
guardados), porque una sola familia en producción es más simple y, si hay empate, no cuesta
precisión.

**Varias alternativas contra el mismo modelo.** El intervalo de 95% controla el error de una
comparación: si la alternativa no es mejor, hay 5% de probabilidad de que lo parezca por azar. Cuando
se prueban k alternativas para responder la misma pregunta (varias variantes, varios pesos de una
mezcla, una grilla de hiperparámetros), la probabilidad de que al menos una parezca mejor por azar
crece con k. Es el problema de comparar muchos pronósticos contra uno de referencia (*data snooping*;
White, 2000; Hansen, 2005). En ese caso cada intervalo se calcula con nivel 1 − 0.05/k, la corrección
de Bonferroni (`experimentos.diferencia_bootstrap(..., nivel=...)`), donde k cuenta todas las
alternativas evaluadas para esa pregunta en los tres resultados. Un intervalo más ancho solo puede
convertir una mejora en empate: lo que ya empata con el intervalo de 95% sigue empatando. Un ajuste
(hiperparámetros, variables, ventana de entrenamiento, mezcla) se adopta solo si mejora la métrica que
decide con un intervalo, ajustado cuando corresponde, que no incluye cero.

**Requisitos para que un modelo sea elegible** (`experimentos.cumple_requisitos`):

- R² fuera de muestra mayor a 0 en validación y en cada año de la evaluación año por año de
  [`5.1_estabilidad_temporal.ipynb`](5.1_estabilidad_temporal.ipynb) (y D² mayor a 0 en touchdowns):
  el modelo tiene que superar al promedio histórico.
- Pendiente de calibración entre 0.9 y 1.1.
- Sesgo adicional de a lo más 5% del promedio real. Cualquier predicción hecha con años anteriores
  hereda el cambio de nivel entre temporadas: la tasa de touchdowns por receptor bajó de 0.216 en
  2016-2021 a 0.193 en 2022-2023, así que incluso predecir la media de entrenamiento tiene un sesgo de
  12% en validación. El requisito mide lo que el modelo agrega encima de ese cambio:
  |sesgo del modelo| − |sesgo de la media de entrenamiento|. Una primera versión pedía sesgo total de
  a lo más 5%, que ni la media de entrenamiento cumple en touchdowns; se corrigió al ver ese
  resultado.

Los umbrales son convención del proyecto: la literatura pide calibración y superar al promedio
histórico, y estos valores fijan cuánto margen se acepta.

In [5]:
print("requisitos:", experimentos.REQUISITOS)
pd.DataFrame(modeling.POLITICA_METRICAS).T

requisitos: {'sesgo_adicional_maximo': 0.05, 'pendiente_minima': 0.9, 'pendiente_maxima': 1.1}


,principal,conteo,evento
receptions,rmse,True,False
receiving_yards,rmse,False,False
receiving_tds,deviance_poisson,True,True


### Un límite propio de los touchdowns

Si los touchdowns de cada receptor siguieran una distribución de Poisson con un valor esperado λ
propio, su varianza total sería Var(λ) + E[λ]: la parte que depende del jugador más el azar de cada
partido. Un modelo que conociera λ exacto explicaría a lo más Var(λ) / Var(touchdowns), que se
aproxima con (varianza − media) / varianza. Es una referencia, no un tope exacto: depende de que el
azar de cada partido sea el de una Poisson. Si es menor, el R² alcanzable es mayor. Por eso se mira
también cómo se reparten los touchdowns por partido.

In [6]:
y_tds = wr["receiving_tds"]
for nombre, mascara in (("entrenamiento", train_mask), ("validacion", val_mask), ("prueba", test_mask)):
    v, mu = y_tds[mascara].var(), y_tds[mascara].mean()
    reparto = y_tds[mascara].clip(upper=2).value_counts(normalize=True).sort_index()
    print(f"{nombre:14s} varianza {v:.4f} | media {mu:.4f} | R2 maximo bajo Poisson = {(v - mu) / v:.3f} | "
          f"partidos con 0, 1 y 2+ touchdowns: {reparto[0]:.1%}, {reparto[1]:.1%}, {reparto[2]:.1%}")

entrenamiento  varianza 0.2304 | media 0.2163 | R2 maximo bajo Poisson = 0.061 | partidos con 0, 1 y 2+ touchdowns: 81.2%, 16.2%, 2.6%
validacion     varianza 0.2055 | media 0.1929 | R2 maximo bajo Poisson = 0.061 | partidos con 0, 1 y 2+ touchdowns: 83.0%, 15.0%, 2.0%
prueba         varianza 0.2182 | media 0.2034 | R2 maximo bajo Poisson = 0.068 | partidos con 0, 1 y 2+ touchdowns: 82.3%, 15.3%, 2.4%


**Lectura.** Bajo el supuesto Poisson, un modelo perfecto llegaría a un R² de 0.061 en entrenamiento
y validación y de 0.068 en prueba. El supuesto exagera el azar de un partido: en 81-83% de los
partidos el receptor no anota, en 15-16% anota uno y solo en 2.0-2.6% anota dos o más. Un conteo que
casi siempre es 0 o 1 varía menos que una Poisson de la misma media (si solo pudiera valer 0 o 1, su
varianza sería p(1 − p), menor que p). El R² alcanzable es entonces algo mayor que esa referencia,
sin que este cálculo diga cuánto.

La conclusión práctica no cambia: la mayor parte de la variación de touchdowns entre partidos es
azar. Un R² de touchdowns cercano a esa referencia, o algo mayor, no indica un mal modelo; uno
negativo sí indica un problema, porque significa que el modelo es peor que el promedio histórico.

## 4. Valores faltantes por familia de modelo

La primera fila de un jugador en la tabla no tiene partidos anteriores de los que promediar. Un
ejemplo real:

In [7]:
primera_fila = wr.sort_values(["player_id", "season", "week"]).groupby("player_id").head(1)
ejemplo = primera_fila[primera_fila["player_display_name"].notna()].iloc[0]
cols_ejemplo = ["player_display_name", "season", "week", "anios_experiencia", "targets_season_avg",
                "targets_last3_avg", "targets_last5_avg", "targets_career_avg", "depth_team"]
print(ejemplo[cols_ejemplo])
print(f"\n% de nulos en cada feature de árbol, sobre toda la tabla:")
print(wr[features_arbol].isna().mean().sort_values(ascending=False).round(3).head(10))

player_display_name    Steve Smith
season                        2016
week                             1
anios_experiencia               15
targets_season_avg             NaN
targets_last3_avg              NaN
targets_last5_avg              NaN
targets_career_avg             NaN
depth_team                     1.0
Name: 0, dtype: object

% de nulos en cada feature de árbol, sobre toda la tabla:
receptions_season_avg                     0.098
receiving_yards_after_catch_season_avg    0.098
targets_season_avg                        0.098
wopr_season_avg                           0.098
target_share_season_avg                   0.098
target_share_x_ofensiva_equipo            0.076
depth_team                                0.075
receptions_career_avg                     0.030
receptions_last5_avg                      0.030
targets_last5_avg                         0.030
dtype: float64


**Lectura.** La fila es de Steve Smith en la semana 1 de 2016. No es un novato
(`anios_experiencia` = 15): 2016 es la primera temporada que cubren los datos y no hay historia
previa que promediar. Un novato real tendría el mismo vacío por otra razón; en ambos casos no hay
nada real que promediar todavía. Los promedios de temporada faltan en 9.8% de las filas, que son la
primera aparición de cada jugador en cada temporada (el promedio de temporada se reinicia).
`depth_team` falta en 7.5%: sobre todo, receptores que esa semana no aparecen en ninguna alineación
de su equipo (ver `1.3_unificacion_alineaciones.ipynb`). La interacción con la ofensiva del equipo
falta en 7.6%, cuando todavía no hay semanas previas del equipo en la temporada o del jugador.

**Manejo por familia de modelo** (dentro de `experimentos.entrenar_y_evaluar`):

- **XGBoost y HistGradientBoosting**: manejan los valores faltantes de forma nativa; el árbol aprende
  hacia qué lado mandarlos en cada división.
- **RandomForest**: scikit-learn no acepta faltantes, así que se imputan con la mediana calculada
  solo en entrenamiento.
- **Ridge**: la misma imputación más escalado (`StandardScaler`), que los notebooks originales de
  `_reference/` no hacían; el porqué está en `4.2_modelos_recepciones_yardas.ipynb`.
- **Baseline**: usa el promedio de temporada del propio jugador, con respaldo en su promedio de
  carrera y después en la media de entrenamiento.

### Referencias

- Gneiting, T. (2011). [Making and evaluating point forecasts](https://arxiv.org/abs/0912.0902).
  *Journal of the American Statistical Association*, 106(494).
- Czado, C., Gneiting, T. y Held, L. (2009).
  [Predictive model assessment for count data](https://www.zora.uzh.ch/entities/publication/3a6b88b2-4bc3-4929-80ef-010bdd2b69ce).
  *Biometrics*, 65(4).
- Gneiting, T. y Resin, J. (2023).
  [Regression diagnostics meets forecast evaluation](https://projecteuclid.org/journals/electronic-journal-of-statistics/volume-17/issue-2/Regression-diagnostics-meets-forecast-evaluation-conditional-calibration-reliability-diagrams/10.1214/23-EJS2180.full).
  *Electronic Journal of Statistics*, 17(2).
- Campbell, J. Y. y Thompson, S. B. (2008).
  [Predicting excess stock returns out of sample](https://nber.org/papers/w11468). *Review of
  Financial Studies*, 21(4).
- Diebold, F. X. y Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business &
  Economic Statistics*, 13(3).
- White, H. (2000). A reality check for data snooping. *Econometrica*, 68(5).
- Hansen, P. R. (2005). A test for superior predictive ability. *Journal of Business & Economic
  Statistics*, 23(4).
- Kolassa, S. (2016). Evaluating predictive count data distributions in retail sales forecasting.
  *International Journal of Forecasting*, 32(3).
- Walsh, C. y Joshi, A. (2024).
  [Machine learning for sports betting: should model selection be based on accuracy or calibration?](https://researchportal.bath.ac.uk/en/publications/machine-learning-for-sports-betting-should-model-selection-be-bas/)
  *Machine Learning with Applications*.
- FantasyPros, [metodología de exactitud semanal](https://fantasypros.com/about/faq/football-inseason-accuracy-methodology).
- scikit-learn, [D² de deviance](https://scikit-learn.org/1.6/modules/generated/sklearn.metrics.d2_tweedie_score.html).

La decisión completa está en
[`docs/decisions/0004-metricas-de-seleccion.md`](../../../docs/decisions/0004-metricas-de-seleccion.md).

## Cierre

La tabla, las variables, las particiones, la política de métricas y el manejo de faltantes quedan
fijos para todo el modelado. Todavía no hay ningún modelo: la comparación empieza en el notebook
siguiente.

Anterior: [`3.3_seleccion_estable.ipynb`](3.3_seleccion_estable.ipynb) · Siguiente:
[`4.2_modelos_recepciones_yardas.ipynb`](4.2_modelos_recepciones_yardas.ipynb)